# Build a model to predict Koc 

The aim of this assignment is to build a model that can predict the soil sorption coefficient logKoc. In contrast to the previous assignment, this time the goal is outperform the state-of-the-art OPERA Koc model:

 - Paper on OPERA models, including the one for Koc: https://doi.org/10.1186/s13321-018-0263-1
 - Performance of the OPERA Koc model on the test set: R2 = 0.71, RMSE = 0.61
 - Model: weighted k-Nearest Neighbors regressor trained on 12 selected PaDEL descriptors (similar to RDKit descriptors)

As you are now a modelling expert, it is open to you which model architecture you use. BUT we ask you to justify your choices! 


#### Tasks:

1) Load the training data and split it into train and test according to the 'Tr_1_Tst_0' column

2) Think about a suitable architecture:
    - Suitable descriptors/fingerprints
    - Model choice (fine-tune a pretrained neural network? GNN? Ensemble method? Gaussian Process? ...?)
    - Consider providing an AD for your model (using an AD metric, or by defining the content of the training data (e.g., organic chemicals with a MW between x and y)
    - Consider providing prediction uncertainty: If you decide to do so, provide uncertainty calibration. If not, explain why.
    
        
3) Train the model on the same training data used in the paper (Tr_1_Tst_0 == 1). 

4) Consider tuning hyperparameters (e.g., using GridSearchCV and on a short list of parameters) 

5) Evaluate model on the test set (Tr_1_Tst_0 == 0), ONLY ONCE ! 

6) Compare your model performance on the test set to the OPERA model. 

#### Questions:
1) Which architecture did you choose, and why?
2) How well does your model perform on the test set? Could you outperform OPERA?
3) Did you add prediction uncertainty, and why? Are the uncertainties well calibrated?
4) Do you provide an AD, and why? How did you define your AD


In [ ]:
# import
import pandas as pd
import numpy as np
from rdkit import Chem

# own imports
from rdkit.Chem import Descriptors

from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    Matern,
    WhiteKernel,
    ConstantKernel
)

from sklearn.metrics import r2_score, mean_squared_error

#### 1. Load training data

In [14]:
# Loading the data from the .sdf file
supplier = Chem.SDMolSupplier("KOC_QR.sdf")
df = pd.DataFrame([
    {
        "SMILES": Chem.MolToSmiles(m),
        **{p: m.GetProp(p) for p in ['preferred_name', 'LogKOC', 'Tr_1_Tst_0']}
    }
    for m in supplier if m is not None
])

In [26]:
# ============================================================
# 2. Train / Test split
# ============================================================

df["LogKOC"] = df["LogKOC"].astype(float)

train_df = df[df["Tr_1_Tst_0"] == "1"].copy()
test_df  = df[df["Tr_1_Tst_0"] == "0"].copy()


# ============================================================
# 3. Extract all RDKit descriptors
# ============================================================

descriptor_names = [desc[0] for desc in Descriptors._descList]

def calc_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    values = []
    for name, func in Descriptors._descList:
        try:
            values.append(func(mol))
        except:
            values.append(np.nan)
    return values

# Calculate descriptors
X_train_df = pd.DataFrame(
    [calc_rdkit_descriptors(s) for s in train_df["SMILES"]],
    columns=descriptor_names
)

X_test_df = pd.DataFrame(
    [calc_rdkit_descriptors(s) for s in test_df["SMILES"]],
    columns=descriptor_names
)


# ============================================================
# 4. Remove problematic descriptors
# ============================================================

print(f"Initial descriptor count: {X_train_df.shape[1]}")

# Remove descriptors containing NaNs
valid_cols = X_train_df.columns[~X_train_df.isna().any()]

X_train_df = X_train_df[valid_cols]
X_test_df  = X_test_df[valid_cols]

# Remove constant descriptors
nunique = X_train_df.nunique()

valid_cols = nunique[nunique > 1].index

X_train_df = X_train_df[valid_cols]
X_test_df  = X_test_df[valid_cols]

print(f"Descriptor count after cleaning: {X_train_df.shape[1]}")

# ============================================================
# 5. Mutual Information feature selection
# ============================================================

y_train = train_df["LogKOC"].values
y_test  = test_df["LogKOC"].values

mi_scores = mutual_info_regression(
    X_train_df,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "descriptor": X_train_df.columns,
    "mi_score": mi_scores
})

mi_df = mi_df.sort_values(
    "mi_score",
    ascending=False
)

# Select top 20 descriptors
top_features = mi_df.head(30)["descriptor"].tolist()

print("Top selected descriptors:")
print(top_features)

X_train = X_train_df[top_features]
X_test  = X_test_df[top_features]


# ============================================================
# 6. Optional: remove highly correlated descriptors
# ============================================================

corr_matrix = X_train.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# drop columns with correlation > 0.95, keeping only the one with the highest MI score
to_drop = set()
for column in upper.columns:
    high_corr = upper[column][upper[column] > 0.95].index.tolist()
    if high_corr:
        to_drop.update(high_corr)


# to_drop = [
#     column for column in upper.columns
#     if any(upper[column] > 0.95)
# ]

X_train = X_train.drop(columns=to_drop)
X_test  = X_test.drop(columns=to_drop)

print("\nRemoved correlated descriptors:")
print(to_drop)

print("\nFinal feature count:")
print(X_train.shape[1])


# ============================================================
# 7. Define GPR pipeline
# ============================================================

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("gpr", GaussianProcessRegressor(
        random_state=42,
        normalize_y=True
    ))
])


# ============================================================
# 8. Define parameter grid
# ============================================================

param_grid = [
    {
        "gpr__kernel": [
            ConstantKernel(1.0) *
            RBF(length_scale=1.0) +
            WhiteKernel()
        ],
        "gpr__alpha": [1e-6, 1e-4, 1e-2],
        "gpr__n_restarts_optimizer": [5, 10]
    },
    {
        "gpr__kernel": [
            ConstantKernel(1.0) *
            Matern(length_scale=1.0, nu=1.5) +
            WhiteKernel()
        ],
        "gpr__alpha": [1e-6, 1e-4, 1e-2],
        "gpr__n_restarts_optimizer": [5, 10]
    }
]


# ============================================================
# 9. Grid Search
# ============================================================

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


# ============================================================
# 10. Best model
# ============================================================

best_model = grid.best_estimator_

print("\nBest parameters:")
print(grid.best_params_)

print("\nBest CV RMSE:")
print(-grid.best_score_)


# ============================================================
# 11. Predictions + uncertainty
# ============================================================

y_pred, y_std = best_model.predict(
    X_test,
    return_std=True
)


# ============================================================
# 12. Final evaluation
# ============================================================

r2 = r2_score(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\nTest performance")
print("-------------------")
print(f"R2   : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")


# ============================================================
# 13. Prediction intervals
# ============================================================

lower = y_pred - 1.96 * y_std
upper = y_pred + 1.96 * y_std

results_df = pd.DataFrame({
    "Experimental": y_test,
    "Predicted": y_pred,
    "Uncertainty_STD": y_std,
    "Lower_95CI": lower,
    "Upper_95CI": upper
})

results_df.head()

Initial descriptor count: 217
Descriptor count after cleaning: 186
Top selected descriptors:
['MolLogP', 'Chi4v', 'Chi4n', 'PEOE_VSA6', 'Chi3v', 'SMR_VSA7', 'HeavyAtomMolWt', 'BertzCT', 'MolMR', 'MaxAbsEStateIndex', 'MaxEStateIndex', 'LabuteASA', 'NumValenceElectrons', 'ExactMolWt', 'Chi2v', 'MolWt', 'Chi3n', 'Chi0', 'FpDensityMorgan1', 'Chi1v', 'Chi0n', 'AvgIpc', 'fr_benzene', 'HeavyAtomCount', 'NumAromaticCarbocycles', 'Chi1', 'Chi2n', 'Chi1n', 'VSA_EState3', 'SlogP_VSA6']

Removed correlated descriptors:
{'NumValenceElectrons', 'MolMR', 'ExactMolWt', 'Chi0', 'Chi3n', 'Chi2n', 'Chi1', 'HeavyAtomMolWt', 'Chi4n', 'HeavyAtomCount', 'Chi0n', 'LabuteASA', 'Chi4v', 'fr_benzene', 'MaxAbsEStateIndex'}

Final feature count:
15
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END gpr__alpha=1e-06, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total time=   1.3s
[CV] END gpr__alpha=1e-06, gpr__kernel=1**2 * RBF(length_scale=1)

,Experimental,Predicted,Uncertainty_STD,Lower_95CI,Upper_95CI
0,1.40,1.775766,0.400516,0.990755,2.560777
1,2.01,2.667769,0.614812,1.462739,3.872800
2,1.85,1.773434,0.569451,0.657311,2.889558
3,5.30,5.365823,0.466843,4.450811,6.280835
4,5.37,5.795049,0.512873,4.789819,6.800279


In [4]:
# The data is pre-split in training and testing:
df.groupby('Tr_1_Tst_0').count()

,SMILES,preferred_name,LogKOC
Tr_1_Tst_0,,,
0,184,184,184
1,544,544,544


In [ ]:
# Convert target to float
df["LogKOC"] = df["LogKOC"].astype(float)

# split into training and testing sets
train_df = df[df['Tr_1_Tst_0'] == '1'].copy()
test_df = df[df['Tr_1_Tst_0'] == '0'].copy()

print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")

Training set size: 544
Testing set size: 184


#### 2. Build a supervised model of your choice

In [9]:
# Encoding molecules from SMILES
from rdkit.Chem import AllChem
from rdkit import DataStructs

def mol_to_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=n_bits
    )

    arr = np.zeros((n_bits,))
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

# Create feature matrices
X_train = np.array([mol_to_fp(s) for s in train_df["SMILES"]])
X_test  = np.array([mol_to_fp(s) for s in test_df["SMILES"]])

y_train = train_df["LogKOC"].values
y_test  = test_df["LogKOC"].values

[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerator
[17:26:39] DEPRECATION WARNING: please use MorganGenerat

In [10]:
# Model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

#### 3. Model performance on test set
--> **TEST ONLY ONCE !**

In [11]:
from sklearn.metrics import r2_score, mean_squared_error

# Predict
y_pred = model.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2   : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")

R2   : 0.585
RMSE : 0.736
